In [1]:
# [CELL 1] STUDENT CONFIGURATION, DATA LOADING, AND FEATURE SCREENING
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Pacudan"
STUDENT_ID = "TUPM-26-1859"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

# Find the SECOM data folder in this folder or any parent folder
WORK_DIR = Path.cwd()
OUTPUT_DIR = WORK_DIR


def find_data_dir():
    for base in [WORK_DIR, *WORK_DIR.parents]:
        if (base / "data" / "secom.data").is_file():
            return base / "data"
    raise FileNotFoundError("data/secom.data not found in this folder or any parent folder.")

DATA_DIR = find_data_dir()
measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw.ndim == 2 and len(labels) == len(raw)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])
is_pass = labels == -1
is_fail = labels == 1

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)
print("Workspace:", WORK_DIR.resolve())
print("Data Folder:", DATA_DIR.resolve())
print("Output Folder:", OUTPUT_DIR.resolve())
print("Measurement Matrix Shape:", raw.shape)
print("Pass Records:", int(is_pass.sum()), "| Fail Records:", int(is_fail.sum()))
# Feature screening shared by the additional exercises
missing_mask = np.isnan(raw)
observed_n = (~missing_mask).sum(axis=0)
obs_min = np.where(missing_mask, np.inf, raw).min(axis=0)
obs_max = np.where(missing_mask, -np.inf, raw).max(axis=0)

has_missing = missing_mask.any(axis=0)
all_missing = observed_n == 0
constant = (observed_n > 0) & (obs_min == obs_max)
usable = ~(all_missing | constant)
usable_idx = np.flatnonzero(usable)
print("Usable Features:", len(usable_idx))


Student: Pacudan TUPM-26-1859
Seed Digit: 9
ID Sum: 31
Name Length: 7
Random Seed: 261859
Verification Value: 226
Workspace: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\Lab7_NumPy_Pandas_Additional_Exercises
Data Folder: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\data
Output Folder: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\Lab7_NumPy_Pandas_Additional_Exercises
Measurement Matrix Shape: (1567, 590)
Pass Records: 1463 | Fail Records: 104
Usable Features: 474


In [2]:
# [CELL 2] EXERCISE 1 - VECTORIZED MEASUREMENT QUALITY ANALYSIS
# Requirement 1: features with missing or constant observed values (screened in Cell 1)

# Requirement 2: select 5 + (SEED_DIGIT % 3) usable features reproducibly
n_select = 5 + (SEED_DIGIT % 3)
rng1 = np.random.default_rng(RANDOM_SEED)
sel = np.sort(rng1.choice(usable_idx, size=n_select, replace=False))
X = raw[:, sel]

# Requirement 3: feature-wise statistics (NumPy)
obs = observed_n[sel]
miss_pct = np.isnan(X).mean(axis=0) * 100
means = np.nanmean(X, axis=0)
medians = np.nanmedian(X, axis=0)
sds = np.nanstd(X, axis=0, ddof=1)

# Requirement 4: robust standardized values (median and MAD, zero MAD handled safely)
mad = np.nanmedian(np.abs(X - medians), axis=0)
zero_mad = mad == 0
safe_mad = np.where(zero_mad, 1.0, mad)
robust_z = np.where(zero_mad, 0.0, 0.6745 * (X - medians) / safe_mad)

# Requirement 5: flag observations with |robust z| > 3.5 (NaN is never flagged)
flagged = np.abs(robust_z) > 3.5
flag_counts = flagged.sum(axis=0)
flag_pct = 100 * flag_counts / obs

# Requirement 6: rank features by flagged percentage and export
q = pd.DataFrame({
    "Feature": feature_names[sel],
    "Column_Index": sel,
    "Observed_Count": obs,
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "Missing_Pct": miss_pct,
    "MAD": mad,
    "Zero_MAD": zero_mad,
    "Flagged_Count": flag_counts,
    "Flagged_Pct": flag_pct,
}).sort_values(["Flagged_Pct", "Feature"], ascending=[False, True]).reset_index(drop=True)
q.insert(0, "Rank", q.index + 1)

quality_path = OUTPUT_DIR / "measurement_quality.csv"
q.to_csv(quality_path, index=False)

# Verification: independent loop-based calculation must match the vectorized counts
loop_counts = []
for j in range(X.shape[1]):
    vals = [v for v in X[:, j] if not np.isnan(v)]
    med = float(np.median(vals))
    m = float(np.median([abs(v - med) for v in vals]))
    cnt = 0
    if m > 0:
        for v in vals:
            if abs(0.6745 * (v - med) / m) > 3.5:
                cnt += 1
    loop_counts.append(cnt)
vectorized_ok = bool(np.array_equal(loop_counts, flag_counts)) and quality_path.is_file()

top_missing = q.sort_values("Missing_Pct", ascending=False).iloc[0]

print("Features With Missing Values:", int(has_missing.sum()))
print("Entirely Missing Features:", int(all_missing.sum()))
print("Constant Features:", int(constant.sum()))
print("Usable Features:", len(usable_idx))
print("Features To Select (5 + SEED_DIGIT % 3):", n_select)
print()
print(q.round(4).to_string(index=False))
print()
print("=" * 70)
print("EXERCISE 1 - ASSESSMENT DATA")
print("=" * 70)
print(f"{'Selected Features':<34}: {', '.join(feature_names[sel])}")
print(f"{'Highest Missing Percentage':<34}: {top_missing['Missing_Pct']:.2f}% ({top_missing['Feature']})")
print(f"{'Feature with Highest Flagged %':<34}: {q.iloc[0]['Feature']} ({q.iloc[0]['Flagged_Pct']:.2f}%)")
print(f"{'Total Flagged Observations':<34}: {int(flag_counts.sum())}")
print(f"{'Vectorized Results Verified':<34}: {vectorized_ok}")
print(f"{'Required Output':<34}: {quality_path.name} (created: {quality_path.is_file()})")
print("=" * 70)

Features With Missing Values: 538
Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474
Features To Select (5 + SEED_DIGIT % 3): 5

 Rank     Feature  Column_Index  Observed_Count   Mean  Median  Sample_SD  Missing_Pct    MAD  Zero_MAD  Flagged_Count  Flagged_Pct
    1 Feature_385           385             852 0.0114  0.0068     0.0144      45.6286 0.0019     False            129      15.1408
    2 Feature_471           471            1561 9.6388  7.3960    10.1741       0.3829 1.8030     False            160      10.2498
    3 Feature_567           567            1294 0.0601  0.0486     0.0328      17.4218 0.0105     False            121       9.3509
    4 Feature_467           467            1563 6.2521  5.6450     8.6737       0.2553 0.8195     False             55       3.5189
    5 Feature_154           154            1564 7.6980  7.5127     5.2392       0.1914 1.5750     False              2       0.1279

EXERCISE 1 - ASSESSMENT DATA
Selected Features          